# PV Fault Classification from Infrared Images

This project evaluates validation-selected logit adjustment for a fixed ImageNet-pretrained ResNet-18 classifier using the public InfraredSolarModules dataset.

Dataset: https://github.com/RaptorMaps/InfraredSolarModules

The study addresses the following research questions:

1. Does validation-selected logit adjustment improve macro-F1, balanced accuracy, and recall for rare PV fault classes?
2. What effect does the adjustment have on overall classification accuracy?
3. Which PV fault classes benefit from the adjustment, and which classes remain difficult to classify?

The analysis uses one documented data split and one trained ResNet-18 checkpoint. Results are presented through quantitative metrics, class-level comparisons, error analysis, and a qualitative model-interpretation example.

## Setup

The required libraries, random seed, paths, device, and experiment settings are defined before the analysis.

In [1]:
import importlib.util
import subprocess
import sys

required = {
    "seaborn": "seaborn>=0.13,<1",
    "sklearn": "scikit-learn>=1.4,<2",
    "tqdm": "tqdm>=4.66,<5",
}
missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *missing,
    ])

In [2]:
import hashlib
import json
import platform
import random
import sys
import time
import urllib.request
import zipfile
from copy import deepcopy
from importlib.metadata import version as package_version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    recall_score,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from tqdm import tqdm

sns.set_theme(style="whitegrid")

In [3]:
SEED = 42
QUICK_RUN = False
IMAGE_SIZE = 128
BATCH_SIZE = 128
EPOCHS = 3 if not QUICK_RUN else 1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
TAU_VALUES = np.arange(0.0, 1.51, 0.1)
BOOTSTRAP_ITERATIONS = 2_000
RARE_CLASSES = {"Diode-Multi", "Hot-Spot", "Hot-Spot-Multi", "Soiling"}


In [ ]:
BASE_DIR = Path("/content") if Path("/content").exists() else Path("/tmp")
WORK_DIR = BASE_DIR / "pv_fault_colab"
DATA_DIR = WORK_DIR / "InfraredSolarModules"
OUTPUT_DIR = WORK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [5]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Selected device: {DEVICE}")

Selected device: mps


In [ ]:
set_seed(SEED)
print(f"Random seeds initialized to {SEED}.")

## Data Preparation

The analysis uses the public InfraredSolarModules dataset, which contains infrared images from 12 PV fault classes.

In [6]:
DATASET_URL = (
    "https://raw.githubusercontent.com/RaptorMaps/InfraredSolarModules/"
    "88e2d1febbcefe401c17ec80b8973f36a02a1653/"
    "2020-02-14_InfraredSolarModules.zip"
)
EXPECTED_SHA256 = "b82c706bc719b045ac4f8930570d81767a8a170d0998ca3e09283b585db05b5e"
ARCHIVE_PATH = WORK_DIR / "InfraredSolarModules.zip"
METADATA_PATH = DATA_DIR / "module_metadata.json"


In [7]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
if not ARCHIVE_PATH.exists():
    print("Downloading official InfraredSolarModules archive...")
    urllib.request.urlretrieve(DATASET_URL, ARCHIVE_PATH)

digest = hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest()
if digest != EXPECTED_SHA256:
    raise RuntimeError(f"Dataset checksum mismatch: {digest}")

print("Dataset archive is present and checksum verified.")

In [ ]:
if not METADATA_PATH.exists():
    destination = WORK_DIR.resolve()
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        for member in archive.infolist():
            member_path = (WORK_DIR / member.filename).resolve()
            if member_path != destination and destination not in member_path.parents:
                raise RuntimeError(
                    f"Unsafe archive member path: {member.filename}"
                )
        archive.extractall(WORK_DIR)

In [ ]:
image_count = len(list((DATA_DIR / "images").glob("*.jpg")))
print(f"Dataset ready: {image_count:,} images.")

### Dataset Inspection

In [8]:
# Load metadata and class labels
with METADATA_PATH.open("r", encoding="utf-8") as handle:
    raw_metadata = json.load(handle)

rows = []
for image_id, record in raw_metadata.items():
    relative_path = Path(record["image_filepath"])
    rows.append({
        "image_id": int(image_id),
        "image_filepath": relative_path.as_posix(),
        "label": record["anomaly_class"],
        "absolute_path": str(DATA_DIR / relative_path),
    })
metadata_df = pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)



print(f"Loaded {len(metadata_df):,} metadata records across {metadata_df['label'].nunique()} classes.")
display(
    metadata_df["label"]
    .value_counts()
    .rename_axis("label")
    .reset_index(name="records")
)


Loaded 20,000 metadata records across 12 classes.


,label,records
0,No-Anomaly,10000
1,Cell,1877
2,Vegetation,1639
3,Diode,1499
4,Cell-Multi,1288
5,Shadowing,1056
6,Cracking,940
7,Offline-Module,827
8,Hot-Spot,249
9,Hot-Spot-Multi,246


In [ ]:
# Inspect metadata structure and one image
print(
    f"Metadata table: {metadata_df.shape[0]:,} rows x "
    f"{metadata_df.shape[1]} columns"
)
print("Columns:", ", ".join(metadata_df.columns))
display(metadata_df.head(10))
display(metadata_df.dtypes.rename("dtype").to_frame())

sample_path = Path(metadata_df.iloc[0]["absolute_path"])
with Image.open(sample_path) as sample_image:
    print(
        f"Example image: size={sample_image.size}, "
        f"mode={sample_image.mode}"
    )

In [9]:
# Define the image-content hash used for duplicate checks
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [10]:
# Verify that all metadata-linked image files are present
missing_paths = [
    path
    for path in metadata_df["absolute_path"]
    if not Path(path).exists()
]
if missing_paths:
    raise FileNotFoundError(
        f"Missing {len(missing_paths)} metadata-linked images."
    )

print(f"Validated {len(metadata_df):,} metadata-linked image paths.")

Validated 20,000 metadata-linked image paths.


In [11]:
# Load previously computed image hashes when available
hash_cache_path = WORK_DIR / "image_sha256.csv"
hash_df = None
if hash_cache_path.exists():
    cached = pd.read_csv(hash_cache_path)
    required_columns = {"image_id", "sha256"}
    expected_ids = set(metadata_df["image_id"])
    cache_is_valid = (
        required_columns.issubset(cached.columns)
        and len(cached) == len(metadata_df)
        and not cached["image_id"].duplicated().any()
        and set(cached["image_id"]) == expected_ids
        and cached["sha256"]
        .astype(str)
        .str.fullmatch(r"[0-9a-f]{64}")
        .all()
    )
    if cache_is_valid:
        hash_df = cached[["image_id", "sha256"]].copy()

cache_status = "valid cache found" if hash_df is not None else "hashing required"
print(f"SHA-256 cache check: {cache_status}.")

SHA-256 cache check: valid cache found.


In [12]:
# Compute hashes for image files that are not already cached
if hash_df is None:
    hash_df = metadata_df[["image_id", "absolute_path"]].copy()
    hash_df["sha256"] = [
        sha256_file(path)
        for path in tqdm(
            hash_df["absolute_path"],
            desc="Hashing images",
        )
    ]
    hash_df[["image_id", "sha256"]].to_csv(
        hash_cache_path,
        index=False,
    )
    print(f"Saved {len(hash_df):,} image hashes to cache.")

In [13]:
# Attach image-content hashes to the metadata
image_df = metadata_df.merge(
    hash_df[["image_id", "sha256"]],
    on="image_id",
    how="left",
    validate="one_to_one",
)
assert image_df["sha256"].notna().all()

print(f"Unique image hashes: {image_df['sha256'].nunique():,}")

Unique image hashes: 19,978


### Duplicate Records

In [14]:
# Identify exact duplicate image groups
duplicate_summary = (
    image_df.groupby("sha256")
    .agg(
        count=("image_id", "size"),
        label_count=("label", "nunique"),
    )
    .query("count > 1")
    .reset_index()
)
conflicting_hashes = set(
    duplicate_summary.loc[
        duplicate_summary["label_count"] > 1,
        "sha256",
    ]
)
duplicate_records = image_df[
    image_df["sha256"].isin(duplicate_summary["sha256"])
]

In [ ]:
print(f"Duplicate groups found: {len(duplicate_summary)}")
print(f"Groups with conflicting labels: {len(conflicting_hashes)}")
display(
    duplicate_summary.sort_values(
        ["label_count", "count"],
        ascending=False,
    ).head(10)
)

In [15]:
# Apply the duplicate-record policy
keep_ids = set(image_df["image_id"])
excluded_rows = []
for sha256, group in duplicate_records.groupby("sha256"):
    ids = sorted(group["image_id"].tolist())
    if sha256 in conflicting_hashes:
        for image_id in ids:
            keep_ids.discard(image_id)
            excluded_rows.append({
                "image_id": image_id,
                "sha256": sha256,
                "reason": "conflicting_duplicate_label",
            })
    else:
        for image_id in ids[1:]:
            keep_ids.discard(image_id)
            excluded_rows.append({
                "image_id": image_id,
                "sha256": sha256,
                "reason": "same_label_duplicate_extra",
            })

In [16]:
# Summarise the records retained for modelling
modelling_df = image_df[image_df["image_id"].isin(keep_ids)].copy()
excluded_df = pd.DataFrame(excluded_rows)

print(f"Exact duplicate groups: {len(duplicate_summary)}")
print(f"Conflicting-label duplicate groups: {len(conflicting_hashes)}")
print(f"Records excluded by duplicate policy: {len(excluded_df)}")
display(duplicate_summary.head(10))

Exact duplicate groups: 22
Conflicting-label duplicate groups: 6
Records excluded by duplicate policy: 28


,sha256,count,label_count
0,262abc63c9edfc2749163b45fe6ba4e53438ce7890e486...,2,1
1,31322a29f5580661a71b2bece539ad93bd52c5f76105d5...,2,2
2,586d3653be12c6377f0f9820a0332e506416ce209785ef...,2,1
3,58817e86a107a0a7245aadb4834d4d92dd9f7559f3f72d...,2,1
4,6a892e426de8c4352b2a30f89a6324e458416056b5d0b8...,2,1
5,715c52c80c6207fa3c1105007132adcd8704e4b1db81b8...,2,1
6,819984dfdb7b8db8a9fd92364984c27c78149a6515f879...,2,1
7,9f3c7dd58fc3e1db1545da20ca5605fafd7736ca6b6121...,2,1
8,a97285f851010305f5b6363fa6be345dddbbb2dc0a4075...,2,1
9,b1cb94dc8a1359b16adcca1c9df33706ab993ac4acbcaf...,2,2


In [17]:
def create_stratified_splits(frame, seed=42):
    rng = np.random.default_rng(seed)
    parts = {"train": [], "validation": [], "test": []}
    for _, group in frame.groupby("label", sort=True):
        shuffled = group.iloc[rng.permutation(len(group))].reset_index(drop=True)
        n_train = int(round(len(shuffled) * 0.70))
        n_validation = int(round(len(shuffled) * 0.15))
        parts["train"].append(shuffled.iloc[:n_train])
        parts["validation"].append(shuffled.iloc[n_train:n_train + n_validation])
        parts["test"].append(shuffled.iloc[n_train + n_validation:])
    return {
        name: pd.concat(group_parts, ignore_index=True).sort_values("image_id").reset_index(drop=True)
        for name, group_parts in parts.items()
    }


### Data Splits

In [18]:
# Create class-stratified train, validation, and test splits
splits = create_stratified_splits(modelling_df, SEED)
if QUICK_RUN:
    limits = {"train": 120, "validation": 40, "test": 40}
    splits = {
        name: pd.concat([
            group.sample(
                min(len(group), limits[name]),
                random_state=SEED,
            )
            for _, group in frame.groupby("label", sort=True)
        ], ignore_index=True)
        for name, frame in splits.items()
    }

print(
    "Split sizes: "
    + ", ".join(
        f"{name}={len(frame):,}"
        for name, frame in splits.items()
    )
)

Split sizes: train=13,981, validation=2,996, test=2,995


In [19]:
# Verify class coverage and duplicate separation across splits
split_hashes = {
    name: set(frame["sha256"])
    for name, frame in splits.items()
}
assert split_hashes["train"].isdisjoint(split_hashes["validation"])
assert split_hashes["train"].isdisjoint(split_hashes["test"])
assert split_hashes["validation"].isdisjoint(split_hashes["test"])
assert (
    set(splits["train"]["label"])
    == set(splits["validation"]["label"])
    == set(splits["test"]["label"])
)

print("Split checks passed: class coverage and content hashes are disjoint.")

Split checks passed: class coverage and content hashes are disjoint.


In [ ]:
# Display split sizes and sample records
split_overview = pd.DataFrame({
    "records": {name: len(frame) for name, frame in splits.items()},
    "classes": {name: frame["label"].nunique() for name, frame in splits.items()},
    "unique_images": {name: frame["image_id"].nunique() for name, frame in splits.items()},
}).rename_axis("split")
display(split_overview)

for name, frame in splits.items():
    print(f"{name.title()} split sample:")
    display(frame[["image_id", "image_filepath", "label"]].head(3))

In [20]:
for name, frame in splits.items():
    frame.drop(
        columns=["absolute_path"],
        errors="ignore",
    ).to_csv(
        OUTPUT_DIR / f"{name}_split.csv",
        index=False,
    )
duplicate_summary.to_csv(
    OUTPUT_DIR / "duplicate_groups.csv",
    index=False,
)
excluded_df.to_csv(
    OUTPUT_DIR / "excluded_duplicate_records.csv",
    index=False,
)

print(f"Saved split manifests and duplicate audit to {OUTPUT_DIR}.")

Saved split manifests and duplicate audit to /tmp/pv_fault_colab/outputs.


In [21]:
summary = {
    "metadata_records": len(metadata_df),
    "unique_image_hashes": int(image_df["sha256"].nunique()),
    "duplicate_groups": len(duplicate_summary),
    "conflicting_duplicate_groups": len(conflicting_hashes),
    "excluded_records": len(excluded_df),
    **{
        f"{name}_records": len(frame)
        for name, frame in splits.items()
    },
}
display(pd.Series(summary, name="value").to_frame())
display(
    pd.crosstab(
        splits["train"]["label"],
        columns="train_count",
    ).sort_values("train_count")
)

,value
metadata_records,20000
unique_image_hashes,19978
duplicate_groups,22
conflicting_duplicate_groups,6
excluded_records,28
train_records,13981
validation_records,2996
test_records,2995


col_0,train_count
label,
Diode-Multi,122
Soiling,143
Hot-Spot-Multi,172
Hot-Spot,174
Offline-Module,575
Cracking,656
Shadowing,732
Cell-Multi,900
Diode,1047


### Data Summary

The metadata contain 20,000 records and 19,978 unique image hashes. The duplicate policy excludes 28 records, leaving 19,972 records for modelling. The final split contains 13,981 training records, 2,996 validation records, and 2,995 test records. All splits contain every class, and no exact image hash appears in more than one split.

The duplicate check identifies exact duplicate files only. It does not establish that visually similar images or images from the same site are absent.